Importing libraries

In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from keras.applications import EfficientNetV2S #type: ignore
from keras.applications.efficientnet_v2 import preprocess_input #type: ignore

Paths

In [4]:
INPUT_DIR = "Yolo_Output"
IMG_SIZE = 224
BATCH_SIZE = 32
OUTPUT_DIR = "Database"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FEATURES = os.path.join(OUTPUT_DIR, "features_efficient.npy")
OUTPUT_LABELS = os.path.join(OUTPUT_DIR, "labels.npy")
OUTPUT_PATHS = os.path.join(OUTPUT_DIR, "paths.npy")


EfficientNet

In [5]:
base_model = EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

image_paths = []
labels = []

for root, _, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(".jpg"):
            img_path = os.path.join(root, file)
            relative_path = os.path.relpath(img_path, INPUT_DIR)
            class_name = relative_path.split(os.sep)[0]
            image_paths.append(img_path)
            labels.append(class_name)

image_paths, labels = zip(*sorted(zip(image_paths, labels)))
image_paths = list(image_paths)
labels = list(labels)
print("Total images found:", len(image_paths))
features = []

for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):
    batch_paths = image_paths[i:i+BATCH_SIZE]
    batch_images = []

    for path in batch_paths:
        img = cv2.imread(path)

        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        batch_images.append(img)

    batch_images = np.array(batch_images)
    batch_images = preprocess_input(batch_images)
    batch_features = base_model.predict(batch_images, verbose=0)
    features.append(batch_features)

features = np.vstack(features)
labels = np.array(labels)
np.save(OUTPUT_FEATURES, features)
np.save(OUTPUT_LABELS, labels)
np.save(OUTPUT_PATHS, np.array(image_paths))
print("\n✅ Feature extraction complete")
print("Feature shape:", features.shape)
print("Labels shape:", labels.shape)

I0000 00:00:1771694869.496150    6663 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3271 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Total images found: 9861


  0%|          | 0/309 [00:00<?, ?it/s]I0000 00:00:1771694874.102987    6742 service.cc:148] XLA service 0x7ddedc001b20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771694874.103010    6742 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2026-02-21 22:57:54.254018: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1771694874.886644    6742 cuda_dnn.cc:529] Loaded cuDNN version 90101
I0000 00:00:1771694882.555395    6742 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
100%|██████████| 309/309 [00:58<00:00,  5.24it/s]


✅ Feature extraction complete
Feature shape: (9861, 1280)
Labels shape: (9861,)
